# 3. Autograd 自动求导

PyTorch 的核心特性：动态计算图 + 自动微分

### 什么是自动求导？

神经网络训练时需要计算梯度（gradient），也就是每个参数对最终损失的影响程度。手动计算梯度非常繁琐，PyTorch 的 Autograd 机制可以自动帮你算。

### 核心概念

- requires_grad=True — 告诉 PyTorch 追踪这个张量的所有运算，之后需要求导
- backward() — 反向传播，自动计算所有梯度
- .grad — 存放计算出来的梯度值
- 计算图：PyTorch 会自动记录运算过程，反向传播时沿着这个图算梯度

In [1]:
import torch

# requires_grad=True 告诉 PyTorch 追踪这个张量的运算
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"x: {x}")
print(f"requires_grad: {x.requires_grad}")

x: tensor([1., 2., 3.], requires_grad=True)
requires_grad: True


### backward() 反向传播计算梯度

下面这个例子：
1. y = x * 2 + 3 — 对 x 做了一个线性变换
2. z = y.mean() — 求均值，得到一个标量（单个数字）
3. z.backward() — 反向传播，计算 dz/dx

为什么需要是标量才能 backward？因为反向传播需要一个最终的损失值（一个数），然后才能算每个参数对这个数的贡献。

In [2]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# 前向传播
y = x * 2 + 3
z = y.mean()

print(f"y = x*2+3: {y}")
print(f"z = y.mean(): {z}")

# 反向传播
z.backward()
print(f"\nx.grad (dz/dx): {x.grad}")
# dz/dx = d(mean(x*2+3))/dx = 2/3，每个元素都一样

y = x*2+3: tensor([5., 7., 9.], grad_fn=<AddBackward0>)
z = y.mean(): 7.0

x.grad (dz/dx): tensor([0.6667, 0.6667, 0.6667])


### 梯度清零 grad.zero_()

PyTorch 的梯度是累加的，不会自动清零。在训练循环中，每次迭代前必须手动清零，否则梯度会不断叠加，导致参数更新方向错误。

这就是为什么训练代码中总会看到 optimizer.zero_grad()，本质上就是在做梯度清零。

In [3]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# 第一次反向传播
y = (x ** 2).sum()
y.backward()
print(f"第1次 backward 后 x.grad: {x.grad}")

# 不清零，再反向传播一次
y = (x ** 2).sum()
y.backward()
print(f"第2次 backward 后 x.grad（没清零，梯度累加了）: {x.grad}")

# 清零后再算
x.grad.zero_()
y = (x ** 2).sum()
y.backward()
print(f"清零后 backward 的 x.grad: {x.grad}")

第1次 backward 后 x.grad: tensor([2., 4., 6.])
第2次 backward 后 x.grad（没清零，梯度累加了）: tensor([ 4.,  8., 12.])
清零后 backward 的 x.grad: tensor([2., 4., 6.])


### torch.no_grad() 和 detach()

- torch.no_grad() — 包裹一段代码，这段代码中的运算不会被追踪。用于模型推理，因为推理时不需要梯度，关闭追踪可以节省内存。
- .detach() — 从计算图中分离出一个张量，返回一个不需要梯度的新张量。用于把张量当普通数据使用（比如画图、保存结果）。

两者区别：no_grad() 是全局开关，detach() 是针对单个张量。

In [4]:
# torch.no_grad() 推理时关闭梯度追踪
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

with torch.no_grad():
    y = x * 2
    print(f"no_grad 中的 y.requires_grad: {y.requires_grad}")  # False

# no_grad 外面又恢复追踪
z = x * 3
print(f"no_grad 外的 z.requires_grad: {z.requires_grad}")    # True

no_grad 中的 y.requires_grad: False
no_grad 外的 z.requires_grad: True


In [5]:
# detach() 从计算图分离张量
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2 + 3

detached = y.detach()
print(f"y.requires_grad: {y.requires_grad}")            # True
print(f"detached.requires_grad: {detached.requires_grad}")  # False

# detach 后的张量就像普通数据，可以安全地转 NumPy
print(f"detach 后可以转 NumPy: {detached.numpy()}")

y.requires_grad: True
detached.requires_grad: False
detach 后可以转 NumPy: [5. 7. 9.]
